In [2]:
from pathlib import Path
from metasmith.agents import Agent
from metasmith.models.libraries import *
from metasmith.models.remote import *

from local.constants import WORKSPACE_ROOT
SOCKEYE_SOURCE = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=64a5c402-05c4-4607-bbad-46a9c2aebd98&origin_path=%2Fhome%2Ftxyliu%2F")
SOCKEYE_SOURCE.endpoint

'64a5c402-05c4-4607-bbad-46a9c2aebd98'

In [3]:
agent_local = Agent(
    home=Source.FromLocal(WORKSPACE_ROOT/"main/local_mock/cache/local_home"),
)

agent_ssh = Agent(
    home=SshSource(
        host="cosmos",
        path="~/workspace/metasmith_home",
    ).AsSource(),
)

agent_slurm = Agent(
    setup_commands=[
        "module load gcc/9.4.0 apptainer/1.3.1",
    ],
    home=SshSource(
        host="sockeye",
        path="~/scratch/metasmith_home",
    ).AsSource(),
    globus_uuid=SOCKEYE_SOURCE.endpoint,
)

agent=agent_local
# agent=agent_ssh
# agent=agent_slurm
agent.Deploy()

2025-03-14_00-24-38  | /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
2025-03-14_00-24-38  | /home/tony
2025-03-14_00-24-38  | >>> mkdir -p $AGENT_HOME
2025-03-14_00-24-38  | >>> mkdir -p /home/tony/.globus
2025-03-14_00-24-38  | >>> mkdir -p /home/tony/.globusonline


2025-03-14_00-24-38 E| mkdir: missing operand
2025-03-14_00-24-38 E| Try 'mkdir --help' for more information.


2025-03-14_00-24-38  | >>> [ -e /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/metasmith.sif ] || apptainer pull /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-03-14_00-24-38  | staged [msm_stub]
2025-03-14_00-24-38  | staged [msm]
2025-03-14_00-24-38  | >>> cd /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home && ./msm api deploy_from_container
2025-03-14_00-24-38  | including dev binds
2025-03-14_00-24-39  | 2025-03-14_00-24-39  | api call to [deploy_from_container] with [{}]
2025-03-14_00-24-39  | 2025-03-14_00-24-39  | deploying to [/ws]
2025-03-14_00-24-39  | 2025-03-14_00-24-39  | deploying relay server to [/ws/relay/msm_relay]
2025-03-14_00-24-39  | 2025-03-14_00-24-39  | deployment complete
2025-03-14_00-24-40  | staged [lib/agent.yml]
2025-03-14_00-24-40  | staged [lib/msm_bootstrap]
2025-03-14_00-24-40  | staged [lib/nextflow_config]
2025-03-14_00-24

In [4]:
CACHE = WORKSPACE_ROOT/"main/local_mock/cache/xgdb_tests"
trlib = TransformInstanceLibrary.Load("./transforms/simple_1")
xgdb = DataInstanceLibrary.Load(CACHE/"test.xgdb")
# refdb = DataInstanceLibrary.Load(CACHE/"ref.xgdb")
types = DataTypeLibrary.Load(WORKSPACE_ROOT/"main/local_mock/prototypes/metagenomics.dev3.yml")

In [5]:
# chinook_ep = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2F").endpoint
# refdb.SaveAs(GlobusSource(endpoint=chinook_ep, path="/Metasmith/ref.xgdb").AsSource())

refdb = DataInstanceLibrary.LoadFrom(
    src=GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fref.xgdb%2F").AsSource(),
    dest=CACHE/"ref.image.xgdb",
    as_image=True,
)
refdb.remote_src

Source(address='globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb', type=SourceType.GLOBUS)

In [6]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[trlib],
    targets=[
        types["orf_annotations"].WithLineage([
            types["contigs"],
            # xgdb["example.fna"].type,
        ]),
    ],
)

print(task.plan._key)
for step in task.plan.steps:
    print(step.transform.name)

dwfuH8Cz
pprodigal
diamond


In [7]:
task.config = dict(
    nextflow = dict(
        # preset = "slurm",
        preset = "default",
    ),
)

In [15]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-03-14_00-39-34  | connecting to deployed agent
2025-03-14_00-39-34  | starting relay service
  | > 2025-03-14_00-39-35  | connecting to relay as [frKgt0lFEIKY]


 E| > 2025-03-14_00-39-35 E| relay server already running in [relay/connections]
2025-03-14_00-39-36 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz]
2025-03-14_00-39-36 W| clearing previously staged task


2025-03-14_00-39-36  | sending metadata for workflow [dwfuH8Cz]
2025-03-14_00-39-37  | staging
  | > including dev binds
  | > 2025-03-14_00-39-38  | api call to [stage_workflow] with [{'task_key': 'dwfuH8Cz'}]
  | > 2025-03-14_00-39-38  | staging workflow [dwfuH8Cz] with [2] data libs and [1] transform libs
  | > 2025-03-14_00-39-38  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
  | > 2025-03-14_00-39-38  | work [/ws/runs/dwfuH8Cz]
  | > 2025-03-14_00-39-38  | data [/msm_home/data]
  | > 2025-03-14_00-39-38  | external work [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz]
  | > 2025-03-14_00-39-38  | external data [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/data]
  | > 2025-03-14_00-39-38  | additional params:
  | > 2025-03-14_00-39-38  |     nextflow:
  | > 2025-03-14_00-39-38  |       preset: default
  | > 2025-03-14_00-39-38  | moving remote data libraries to [/msm_home/data]
  | > 2025-03

In [16]:
# import shutil
# work_root = WORKSPACE_ROOT/"main/local_mock/cache/local_home/runs/kCvaS6w9"
# for p in [".nextflow", "nxf_logs", "nxf_work", "results"]:
#     shutil.rmtree(work_root/p, ignore_errors=True)
# shutil.rmtree(WORKSPACE_ROOT/"main/local_mock/mock/cache", ignore_errors=True)
    
agent.RunWorkflow(task)

2025-03-14_00-39-48  | connecting to deployed agent
2025-03-14_00-39-48  | starting relay service
  | > 2025-03-14_00-39-49  | connecting to relay as [hzKpq44sywlx]


 E| > 2025-03-14_00-39-49 E| relay server already running in [relay/connections]


2025-03-14_00-39-49  | executing workflow
  | > including dev binds
  | > 2025-03-14_00-39-50  | api call to [execute_workflow] with [{'key': 'dwfuH8Cz'}]
  | > 2025-03-14_00-39-50  | workspace [/msm_home/runs/dwfuH8Cz]
  | > 2025-03-14_00-39-50  | external workspace [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz]
  | > 2025-03-14_00-39-50  | executing workflow [dwfuH8Cz] with preset [default]
  | > 2025-03-14_00-39-50  | preset [default]
  | > 2025-03-14_00-39-50  | steps [2]
  | > 2025-03-14_00-39-50  | locating input data with personal globus endpoint
  | > 2025-03-14_00-39-50  | [3RJW32jRo2ju] is at [/msm_home/runs/dwfuH8Cz/_metasmith/task/transforms/3RJW32jRo2ju]
  | > 2025-03-14_00-39-50  | [gzLTT7PL67JN] is at [/msm_home/runs/dwfuH8Cz/_metasmith/task/data/gzLTT7PL67JN]
  | > 2025-03-14_00-39-50  | [zHXmpWcrgYaH] at [/msm_home/data/zHXmpWcrgYaH] is remote [globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb], downloading to [/ho

In [ ]:
import mimetypes

def istext(filename):
    s=open(filename, encoding="latin1").read(512)
    text_characters = "".join([chr(x) for x in range(32, 127)] + list("\n\r\t\b"))
    translation_table = str.maketrans("", "", text_characters)
    if not s:
        # Empty files are considered text
        return True
    if "\0" in s:
        # Files with null bytes are likely binary
        return False
    # Get the non-text characters (maps a character to itself then
    # use the 'remove' option to get rid of the text characters.)
    t = s.translate(translation_table)
    # If more than 30% non-text characters, then
    # this is considered a binary file
    if float(len(t))/float(len(s)) > 0.30:
        return False
    return True

# istext("/home/tony/workspace/tools/Metasmith/metasmith.sif")
istext("/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz/nxf_work/dd/6d7e3eef979ec8010613bb376b63b6/container.diamond.oci.uri")

True